# 🧩 Pokémon Vision Lab: PCA from pixels to patterns

### Our question
**Can PCA compress Pokémon images, reveal visual patterns, and help a simple classifier recognise the species?**

We will build one small, honest experiment and implement PCA twice:

1. **From scratch with NumPy** so we can see the mathematics.
2. **With scikit-learn** so we can compare our implementation with a trusted library.

### The journey
`JPEG images → consistent pixels → data matrix → centre → covariance → eigenvectors → principal components → visual map → reconstruction → classification`

> **Important:** PCA is an *unsupervised dimensionality-reduction method*. It does not classify Pokémon by itself. We will use a separate, deliberately simple **nearest-centroid classifier** on the PCA features.

### Before running
Arrange your images in class folders, for example:

```text
PCA-pokemon/
├── pokemon_images/
│   ├── Bulbasaur/
│   │   ├── bulbasaur_01.jpg
│   │   └── bulbasaur_02.jpg
│   ├── Charmander/
│   │   ├── charmander_01.jpg
│   │   └── charmander_02.jpg
│   └── Pikachu/
│       ├── pikachu_01.jpg
│       └── pikachu_02.jpg
└── Pokémon_PCA_Lab.ipynb
```

Use as many **genuine, different images per species** as you can. Aim for at least 8–10 originals per class if possible. The notebook also supports a flat folder if filenames start with the class name, such as `pikachu_01.jpg`.

**Fair-test rule:** We split the original images first. Only training images are augmented. Test images are never used to fit PCA or to create training variants.

## 0. What will we need?

We use:
- **Pillow** to open, resize and augment images.
- **NumPy** for the PCA mathematics.
- **Matplotlib** to visualise images, variance and the PCA map.
- **scikit-learn** only for the comparison implementation and evaluation utilities.

If a package is missing, install it in your terminal:

```bash
python -m pip install numpy matplotlib pillow scikit-learn
```

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageOps, ImageEnhance
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Small images keep the from-scratch covariance calculation fast and explainable.
IMAGE_SIZE = 16                 # 16 x 16 pixels
IMAGE_CHANNELS = 3              # RGB
N_FEATURES = IMAGE_SIZE * IMAGE_SIZE * IMAGE_CHANNELS

print("Each image will become", N_FEATURES, "numbers.")

## 1. Find and load the Pokémon images

**What happens here?** A computer does not understand a picture as “Pikachu”. It sees a grid of pixel values. We first make every image the same size and colour format so their numbers can be compared.

We use a square canvas and preserve the image's aspect ratio. Transparent images are placed on a white background.

If your folder is not found, change `DATA_DIR` below. The loader accepts:
- **Class folders**: `pokemon_images/Pikachu/*.jpg`
- **Flat files**: `pokemon_images/pikachu_01.jpg` (the filename must begin with a class name)

For flat files, edit `FLAT_LABEL_PREFIXES` to match your species.

In [ ]:
# EDIT THIS if your image folder has a different name or location.
possible_dirs = [
    Path("pokemon_images"),
    Path("data/pokemon_images"),
    Path("images"),
    Path(".") / "PCA-pokemon" / "pokemon_images",
]
DATA_DIR = next((p for p in possible_dirs if p.exists() and p.is_dir()), possible_dirs[0])
print("Image folder:", DATA_DIR.resolve())

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Used only when images are directly inside pokemon_images/ (not in class folders).
# Add or change prefixes to match your filenames.
FLAT_LABEL_PREFIXES = ["bulbasaur", "charmander", "pikachu", "squirtle"]

def discover_images(root):
    records = []
    if not root.exists():
        raise FileNotFoundError(
            f"Could not find {root.resolve()}. Put your images in a 'pokemon_images' folder "
            "next to this notebook, or update DATA_DIR."
        )

    # Preferred format: one subfolder per species.
    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if class_dirs:
        for class_dir in class_dirs:
            for path in sorted(class_dir.rglob("*")):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    records.append((path, class_dir.name))
    else:
        # Alternative format: class name is the beginning of each filename.
        for path in sorted(root.iterdir()):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                lower_name = path.stem.lower()
                label = next(
                    (name.title() for name in FLAT_LABEL_PREFIXES
                     if lower_name.startswith(name)),
                    None
                )
                if label is not None:
                    records.append((path, label))

    if not records:
        raise ValueError(
            "No labelled images found. Use one folder per Pokémon species, "
            "or make flat filenames start with a label in FLAT_LABEL_PREFIXES."
        )
    return records

records = discover_images(DATA_DIR)
labels_found = sorted({label for _, label in records})
print(f"Found {len(records)} images across {len(labels_found)} classes:")
for label in labels_found:
    print(f"  {label}: {sum(y == label for _, y in records)} images")

### 1A. Make every image comparable

We resize to **16 × 16 RGB**. That is intentionally small: it gives us 768 pixel features per image, which is manageable for explaining the covariance matrix and eigenvectors live.

The trade-off: small images lose fine details. That is useful for teaching compression, but it may also make classification harder.

In [ ]:
def load_rgb_vector(path, size=IMAGE_SIZE):
    with Image.open(path) as im:
        # Composite transparency over white, then convert to RGB.
        rgba = im.convert("RGBA")
        white = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        rgb = Image.alpha_composite(white, rgba).convert("RGB")

        # Fit inside a square without stretching; fill unused space with white.
        square = ImageOps.pad(
            rgb, (size, size),
            method=Image.Resampling.LANCZOS,
            color=(255, 255, 255),
            centering=(0.5, 0.5)
        )
        arr = np.asarray(square, dtype=np.float32) / 255.0

    return arr.reshape(-1), arr

image_vectors = []
image_labels = []
image_paths = []

for path, label in records:
    try:
        vector, _ = load_rgb_vector(path)
        image_vectors.append(vector)
        image_labels.append(label)
        image_paths.append(path)
    except Exception as exc:
        print(f"Skipping unreadable image {path}: {exc}")

X_original = np.asarray(image_vectors, dtype=np.float32)
y_original = np.asarray(image_labels)

print("Dataset matrix shape:", X_original.shape)
print("Rows = images; columns = pixel features.")

### 1B. Look at the dataset before doing maths

This gallery is a sanity check. Are images recognisable? Are they consistently framed? Does one class have far fewer images? PCA cannot fix a poorly prepared dataset.

In [ ]:
def show_gallery(X, y, paths=None, max_per_class=5):
    classes = sorted(np.unique(y))
    fig, axes = plt.subplots(len(classes), max_per_class,
                             figsize=(2.1 * max_per_class, 2.2 * len(classes)))
    if len(classes) == 1:
        axes = np.array([axes])
    for row, label in enumerate(classes):
        indices = np.where(y == label)[0][:max_per_class]
        for col in range(max_per_class):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(indices):
                idx = indices[col]
                ax.imshow(X[idx].reshape(IMAGE_SIZE, IMAGE_SIZE, 3))
                if col == 0:
                    ax.set_title(str(label), loc="left", fontsize=10)
    plt.suptitle("A sample of our original Pokémon images")
    plt.tight_layout()
    plt.show()

show_gallery(X_original, y_original)

## 2. Split the original images before augmentation

We reserve some **original** images as a test set. The model must not see these images while learning its PCA directions or class centroids.

For a tiny dataset, a test score will be unstable. Try to collect more original images rather than relying on augmentation to pretend there are more independent examples.

In [ ]:
class_counts = {label: int(np.sum(y_original == label)) for label in np.unique(y_original)}
if len(class_counts) < 2:
    raise ValueError("At least two Pokémon species are needed for classification.")
if min(class_counts.values()) < 2:
    raise ValueError("Each species needs at least 2 original images for a train/test split.")

# Stratify keeps each species represented in both sets.
X_train_orig, X_test, y_train_orig, y_test, paths_train, paths_test = train_test_split(
    X_original, y_original, image_paths,
    test_size=max(0.25, len(class_counts) / len(y_original)),
    random_state=RANDOM_STATE,
    stratify=y_original
)

print("Original training images:", len(X_train_orig))
print("Held-out test images:", len(X_test))
print("Training classes:", sorted(np.unique(y_train_orig)))

## 3. Add controlled variety to training images only

Why augment? A Pokémon may appear slightly brighter, darker, tilted or shifted. We can create modest variations to help the training set represent these changes.

We keep the original training image and create a few altered versions. **We do not augment the test set.** These variants are not new independent Pokémon photographs, so they must not be counted as extra test examples.

In [ ]:
def augment_image(image_array):
    # Convert the already-standardised RGB array back to a PIL image.
    base = Image.fromarray(np.uint8(np.clip(image_array, 0, 1) * 255))
    variants = []

    # Small brightness changes.
    variants.append(np.asarray(ImageEnhance.Brightness(base).enhance(0.85)) / 255.0)
    variants.append(np.asarray(ImageEnhance.Brightness(base).enhance(1.15)) / 255.0)

    # Small rotations; white corners keep the canvas consistent.
    variants.append(np.asarray(
        base.rotate(8, resample=Image.Resampling.BICUBIC,
                    fillcolor=(255, 255, 255))
    ) / 255.0)
    variants.append(np.asarray(
        base.rotate(-8, resample=Image.Resampling.BICUBIC,
                    fillcolor=(255, 255, 255))
    ) / 255.0)

    return [v.astype(np.float32) for v in variants]

X_train_images = [row.reshape(IMAGE_SIZE, IMAGE_SIZE, 3) for row in X_train_orig]
X_train_aug = []
y_train_aug = []
is_augmented = []

for img, label in zip(X_train_images, y_train_orig):
    # Keep the original training image.
    X_train_aug.append(img.reshape(-1))
    y_train_aug.append(label)
    is_augmented.append(False)

    # Add four controlled variants of this training image.
    for variant in augment_image(img):
        X_train_aug.append(variant.reshape(-1))
        y_train_aug.append(label)
        is_augmented.append(True)

X_train = np.asarray(X_train_aug, dtype=np.float64)
y_train = np.asarray(y_train_aug)
is_augmented = np.asarray(is_augmented)

print("Training matrix after augmentation:", X_train.shape)
print("Held-out test matrix (unchanged):", X_test.shape)

### See what augmentation actually changed

This is a useful presentation moment: ask the class which changes preserve the identity and which changes might confuse a pixel-based method.

In [ ]:
# Display one training image and its generated variants.
example = X_train_orig[0].reshape(IMAGE_SIZE, IMAGE_SIZE, 3)
example_variants = augment_image(example)
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
axes[0].imshow(example); axes[0].set_title("Original")
for ax, variant, title in zip(
    axes[1:], example_variants,
    ["Darker", "Brighter", "Rotate +8°", "Rotate -8°"]
):
    ax.imshow(variant); ax.set_title(title)
for ax in axes: ax.axis("off")
plt.suptitle("Training-only augmentation")
plt.tight_layout()
plt.show()

## 4. PCA from scratch: first understand the data matrix

Let:
- **n** = number of training images (including training-only variants)
- **d** = number of pixel features (768)

Our data matrix has shape **n × d**. Each row is one image. Each column is one colour value at one pixel position.

PCA is sensitive to the scale and alignment of the inputs. Here every pixel is scaled to 0–1 and all images use the same resize and padding procedure.

In [ ]:
print(f"n training samples = {X_train.shape[0]}")
print(f"d pixel features   = {X_train.shape[1]}")
print(f"X_train shape      = {X_train.shape}")

## 5. LIVE MATHEMATICS ①: Centre every pixel feature

PCA looks for variation **around the average image**.

For each pixel feature, calculate its mean across training images. Then subtract that mean from every training image.

\[
X_{centered} = X_{train} - \mu
\]

We calculate the mean using training data only, then reuse it for test images.

In [ ]:
# ⭐ LIVE-CODE: pause here and explain each operation to the class.
# LIVE-CODE ①: calculate the average pixel value for every feature.
mean_image = X_train.mean(axis=0)

# LIVE-CODE ①: subtract the average image from every training image.
X_centered = X_train - mean_image

# The test set must use the SAME training mean.
X_test_centered = X_test.astype(np.float64) - mean_image

print("Mean vector:", mean_image.shape)
print("Centred training matrix:", X_centered.shape)

### What does the mean image look like?

The mean image is not a Pokémon. It is the average colour at each pixel position across all training examples. The centred images describe how each image differs from that average.

In [ ]:
mean_rgb = np.clip(mean_image.reshape(IMAGE_SIZE, IMAGE_SIZE, 3), 0, 1)
plt.figure(figsize=(3, 3))
plt.imshow(mean_rgb)
plt.title("Mean Pokémon image")
plt.axis("off")
plt.show()

## 6. LIVE MATHEMATICS ②: Covariance matrix

Covariance describes how two pixel features vary together.

\[
C = \frac{X_{centered}^{T}X_{centered}}{n-1}
\]

Because we have **d = 768** features, the covariance matrix is **768 × 768**. Its diagonal contains each feature's variance; off-diagonal entries describe relationships between pairs of features.

We use `@` for matrix multiplication and `.T` for transpose.

In [ ]:
# ⭐ LIVE-CODE: pause here and explain each operation to the class.
# LIVE-CODE ②: build the feature-by-feature covariance matrix.
n_train = X_centered.shape[0]
covariance = (X_centered.T @ X_centered) / (n_train - 1)

print("Covariance matrix shape:", covariance.shape)

## 7. LIVE MATHEMATICS ③: Eigenvalues and eigenvectors

An eigenvector gives a direction in feature space. Its eigenvalue tells us how much variance lies along that direction.

PCA keeps the directions with the **largest eigenvalues**.

The covariance matrix is symmetric, so `np.linalg.eigh` is the appropriate NumPy routine. We sort the results from largest eigenvalue to smallest.

In [ ]:
# ⭐ LIVE-CODE: pause here and explain each operation to the class.
# LIVE-CODE ③: find eigenvalues and eigenvectors of the covariance matrix.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)

# Sort principal directions from most variance to least variance.
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Tiny negative values can occur from floating-point rounding.
eigenvalues = np.maximum(eigenvalues, 0)

total_variance = eigenvalues.sum()
if total_variance <= 0:
    raise ValueError("The dataset has no measurable variation. Check that images are not identical.")

explained_variance_ratio = eigenvalues / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

print("Eigenvectors shape:", eigenvectors.shape)
print("First 5 eigenvalues:", eigenvalues[:5])
print("Variance explained by first 5 PCs:", explained_variance_ratio[:5])

### 7A. Scree plot: how many components should we keep?

A principal component is one new feature. PC1 captures the most variance, PC2 the next most, and so on.

The cumulative curve answers: “If we keep the first k components, what fraction of the training-set variance do they retain?”

**Variance retained is not the same as classification accuracy.** A direction with lots of visual variation is not automatically the most useful direction for distinguishing species.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cumulative_variance) + 1),
         cumulative_variance, marker="o", markersize=3)
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.axhline(0.95, linestyle=":", label="95% variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("How much variance do we retain?")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

for target in [0.80, 0.90, 0.95]:
    count = int(np.searchsorted(cumulative_variance, target) + 1)
    print(f"Components needed for {target:.0%} variance: {count}")

## 8. LIVE MATHEMATICS ④: Project images into a smaller space

We select the first **k** eigenvectors and put them into a projection matrix \(W_k\).

\[
Z = X_{centered}W_k
\]

The original image had 768 values. If `k = 20`, its new PCA representation has only 20 values.

This is dimensionality reduction: fewer numbers represent each image.

In [ ]:
# ⭐ LIVE-CODE: pause here and explain each operation to the class.
# LIVE-CODE ④: choose how many principal components to keep.
K = min(20, eigenvectors.shape[1], X_train.shape[0])

# Each column of W_k is one retained principal direction.
W_k = eigenvectors[:, :K]

# Project training and test images using the same learned directions.
Z_train = X_centered @ W_k
Z_test = X_test_centered @ W_k

print("Before PCA:", X_train.shape)
print("After PCA: ", Z_train.shape)
print("Test after PCA:", Z_test.shape)

## 9. What do the principal components look like?

Each eigenvector has 768 values, just like a flattened RGB image. We can reshape it to 16 × 16 × 3 to inspect the pixel pattern.

For display only, we rescale each component's values to 0–1. This display rescaling does **not** change the PCA calculations.

A component image is not a Pokémon photograph. It visualises a direction of pixel variation around the mean.

In [ ]:
n_show = min(6, eigenvectors.shape[1])
fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.5))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    component = eigenvectors[:, i].reshape(IMAGE_SIZE, IMAGE_SIZE, 3)
    lo, hi = component.min(), component.max()
    display_component = (component - lo) / (hi - lo + 1e-12)
    ax.imshow(display_component)
    ax.set_title(f"PC{i+1}\n{explained_variance_ratio[i]:.1%}")
    ax.axis("off")

plt.suptitle("First principal-component patterns (display-normalised)")
plt.tight_layout()
plt.show()

## 10. The Pokémon map: do similar images land near each other?

We plot the first two PCA coordinates. Every dot represents one image variant or held-out original image.

- **PC1** is the direction with the most variance.
- **PC2** is the next direction, perpendicular to PC1.
- Colours show the known species labels. PCA itself did not receive these labels.

Training variants are circles; held-out test images are crosses. The map is a 2D view of a much larger feature space, so overlap is possible and does not automatically mean PCA failed.

In [ ]:
if Z_train.shape[1] >= 2:
    # Use the first two principal coordinates for a 2D map.
    plt.figure(figsize=(9, 6))
    classes = sorted(np.unique(y_train))
    palette = plt.cm.get_cmap("tab10", len(classes))

    for idx, label in enumerate(classes):
        train_mask = (y_train == label)
        test_mask = (y_test == label)

        # Original training images: larger circles.
        original_mask = train_mask & (~is_augmented)
        augmented_mask = train_mask & is_augmented

        plt.scatter(Z_train[original_mask, 0], Z_train[original_mask, 1],
                    color=palette(idx), marker="o", s=55,
                    edgecolor="black", linewidth=0.4, label=f"{label} train")
        plt.scatter(Z_train[augmented_mask, 0], Z_train[augmented_mask, 1],
                    color=palette(idx), marker=".", s=18, alpha=0.35)
        plt.scatter(Z_test[test_mask, 0], Z_test[test_mask, 1],
                    color=palette(idx), marker="x", s=75, linewidth=2,
                    label=f"{label} test")

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Pokémon images projected onto the first two principal components")
    plt.grid(alpha=0.2)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough principal components for a 2D plot.")

## 11. LIVE MATHEMATICS ⑤: Reconstruct an image

Projection compresses an image. Reconstruction maps its reduced representation back into the original pixel space:

\[
\hat{X} = ZW_k^T + \mu
\]

The reconstructed image is an approximation. With fewer components, more information is discarded. With more components, more detail can be recovered.

We compare several component counts using the **same held-out test image**.

In [ ]:
# ⭐ LIVE-CODE: pause here and explain each operation to the class.
# LIVE-CODE ⑤: reconstruct one held-out image using the first K components.
def reconstruct_with_k(image_vector, mean_vector, components, k):
    centered = image_vector - mean_vector
    W = components[:, :k]
    scores = centered @ W
    reconstruction = scores @ W.T + mean_vector
    return np.clip(reconstruction, 0, 1)

example_index = 0
component_counts = [1, 2, 5, 10, 20, 50]
component_counts = [k for k in component_counts if k <= eigenvectors.shape[1]]

original = X_test[example_index].reshape(IMAGE_SIZE, IMAGE_SIZE, 3)
fig, axes = plt.subplots(1, len(component_counts) + 1,
                         figsize=(2.2 * (len(component_counts) + 1), 2.7))
axes[0].imshow(original)
axes[0].set_title("Original test image")
axes[0].axis("off")

for ax, k in zip(axes[1:], component_counts):
    rebuilt = reconstruct_with_k(
        X_test[example_index].astype(np.float64),
        mean_image, eigenvectors, k
    ).reshape(IMAGE_SIZE, IMAGE_SIZE, 3)
    ax.imshow(rebuilt)
    ax.set_title(f"{k} PCs")
    ax.axis("off")

plt.suptitle("Reconstruction as we keep more principal components")
plt.tight_layout()
plt.show()

### 11A. Measure reconstruction error

We use mean squared error (MSE): the average squared difference between original and reconstructed pixel values. Lower is better for reconstruction.

In [ ]:
reconstruction_rows = []
for k in component_counts:
    rebuilt = np.array([
        reconstruct_with_k(x.astype(np.float64), mean_image, eigenvectors, k)
        for x in X_test
    ])
    mse = np.mean((X_test - rebuilt) ** 2)
    reconstruction_rows.append((k, mse))

print("Components | Test reconstruction MSE")
for k, mse in reconstruction_rows:
    print(f"{k:10d} | {mse:.6f}")

plt.figure(figsize=(6, 4))
plt.plot([r[0] for r in reconstruction_rows],
         [r[1] for r in reconstruction_rows], marker="o")
plt.xlabel("Number of components")
plt.ylabel("Test reconstruction MSE")
plt.title("More components usually reduce reconstruction error")
plt.grid(alpha=0.25)
plt.show()

## 12. Can PCA features help classify a Pokémon?

Now we add a separate, simple classifier: **nearest centroid**.

1. For each species, calculate the average PCA feature vector of its training images.
2. For a test image, measure its Euclidean distance to each species centroid.
3. Predict the species with the smallest distance.

This is intentionally simple so we can focus on what PCA contributes. The classifier receives PCA coordinates, not raw pixels.

In [ ]:
def fit_centroids(Z, labels):
    classes = np.unique(labels)
    centroids = {
        label: Z[labels == label].mean(axis=0)
        for label in classes
    }
    return centroids

def predict_nearest_centroid(Z, centroids):
    classes = list(centroids.keys())
    centre_matrix = np.vstack([centroids[label] for label in classes])
    # One row per test image, one column per class centroid.
    distances = np.linalg.norm(Z[:, None, :] - centre_matrix[None, :, :], axis=2)
    return np.asarray(classes)[np.argmin(distances, axis=1)]

centroids_raw = fit_centroids(Z_train, y_train)
predictions_raw = predict_nearest_centroid(Z_test, centroids_raw)

print("Actual:   ", y_test)
print("Predicted:", predictions_raw)
print(f"Test accuracy: {accuracy_score(y_test, predictions_raw):.1%}")

### 12A. Inspect mistakes, not just the accuracy

A confusion matrix shows which classes are being mixed up. With a small dataset, each test image matters a lot, so always show the number of test examples alongside the percentage.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions_raw,
    labels=sorted(np.unique(y_original)),
    xticks_rotation=35, cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("NumPy PCA + nearest-centroid classifier")
plt.tight_layout()
plt.show()

print(classification_report(y_test, predictions_raw, zero_division=0))

## 13. Compare our PCA with scikit-learn PCA

Now the same experiment is repeated with `sklearn.decomposition.PCA`.

**Fair comparison checklist:**
- Same preprocessed training images.
- Same training-only augmentations.
- Same held-out test images.
- Same number of components.
- Same nearest-centroid classifier.

The library learns its mean from `fit()`. We use `transform()` for both training and test images. The PCA directions may have opposite signs and still be mathematically equivalent.

In [ ]:
# Fit scikit-learn PCA on exactly the same training matrix.
sklearn_pca = PCA(n_components=K, svd_solver="full", random_state=RANDOM_STATE)
Z_train_sklearn = sklearn_pca.fit_transform(X_train)
Z_test_sklearn = sklearn_pca.transform(X_test)

# Apply the SAME nearest-centroid classifier to sklearn PCA coordinates.
centroids_sklearn = fit_centroids(Z_train_sklearn, y_train)
predictions_sklearn = predict_nearest_centroid(Z_test_sklearn, centroids_sklearn)

print("NumPy PCA accuracy:      ", f"{accuracy_score(y_test, predictions_raw):.1%}")
print("scikit-learn PCA accuracy:", f"{accuracy_score(y_test, predictions_sklearn):.1%}")
print("NumPy retained variance: ", f"{explained_variance_ratio[:K].sum():.4f}")
print("sklearn retained variance:", f"{sklearn_pca.explained_variance_ratio_.sum():.4f}")

### 13A. Compare reconstruction error

We compare each implementation's reconstruction of the same held-out test images. Small numerical differences are normal.

In [ ]:
# NumPy reconstruction uses our manually calculated mean and eigenvectors.
X_test_rebuilt_numpy = np.array([
    reconstruct_with_k(x.astype(np.float64), mean_image, eigenvectors, K)
    for x in X_test
])

# sklearn inverse_transform returns the reconstruction in the original feature space.
X_test_rebuilt_sklearn = np.clip(
    sklearn_pca.inverse_transform(Z_test_sklearn), 0, 1
)

mse_numpy = np.mean((X_test - X_test_rebuilt_numpy) ** 2)
mse_sklearn = np.mean((X_test - X_test_rebuilt_sklearn) ** 2)

print(f"NumPy PCA reconstruction MSE:       {mse_numpy:.8f}")
print(f"scikit-learn PCA reconstruction MSE: {mse_sklearn:.8f}")

## 14. Does keeping more components improve classification?

This is a useful final experiment. We vary the number of principal components and repeat the same nearest-centroid classification.

Notice the distinction:
- More components generally preserve more training variance and improve reconstruction.
- More components do **not** guarantee better classification. Extra components may preserve variation that is irrelevant to species identity or may capture noise.

The test set stays fixed throughout this experiment.

In [ ]:
candidate_ks = sorted(set(
    k for k in [1, 2, 5, 10, 20, 40, 80]
    if k <= eigenvectors.shape[1]
))

accuracy_rows = []
for k in candidate_ks:
    W = eigenvectors[:, :k]
    z_train_k = X_centered @ W
    z_test_k = X_test_centered @ W
    centres_k = fit_centroids(z_train_k, y_train)
    pred_k = predict_nearest_centroid(z_test_k, centres_k)
    accuracy_rows.append((k, accuracy_score(y_test, pred_k)))

print("k components | test accuracy")
for k, score in accuracy_rows:
    print(f"{k:12d} | {score:.1%}")

plt.figure(figsize=(7, 4))
plt.plot([r[0] for r in accuracy_rows],
         [r[1] for r in accuracy_rows], marker="o")
plt.xlabel("Number of principal components")
plt.ylabel("Held-out test accuracy")
plt.title("Classification accuracy vs number of PCA components")
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.show()

## 15. Try your own Pokémon image

Place a new image file somewhere accessible and set `NEW_IMAGE_PATH`. The notebook will apply the exact same preprocessing, project it using the **training mean and PCA directions**, then predict the nearest training centroid.

This is a demonstration, not proof that the model will generalise to arbitrary internet images. Backgrounds, scale, pose and image style can change the result.

In [ ]:
# Change this to the path of a new image you want to test.
NEW_IMAGE_PATH = None  # Example: "pokemon_images/Pikachu/new_pikachu.jpg"

if NEW_IMAGE_PATH is not None:
    new_vector, new_rgb = load_rgb_vector(NEW_IMAGE_PATH)
    new_score = (new_vector.astype(np.float64) - mean_image) @ W_k
    new_prediction = predict_nearest_centroid(
        new_score.reshape(1, -1), centroids_raw
    )[0]

    plt.figure(figsize=(3, 3))
    plt.imshow(new_rgb)
    plt.title(f"Predicted: {new_prediction}")
    plt.axis("off")
    plt.show()
else:
    print("Set NEW_IMAGE_PATH to try a new Pokémon image.")

## 16. What did we learn?

Fill in these answers as a team after running the notebook.

1. **Why resize images?** So each image has the same number and arrangement of pixel features.
2. **Why flatten?** To represent each image as one numerical row in a data matrix.
3. **Why centre?** PCA studies variation around the average feature values.
4. **What does covariance tell us?** How pairs of pixel features vary together.
5. **What do eigenvectors represent?** Directions in feature space; the largest eigenvalues identify the directions with the most variance.
6. **What is a principal component score?** An image's coordinate along a learned principal direction.
7. **Why does reconstruction become clearer with more components?** More of the original variation is retained.
8. **Does PCA classify?** No. PCA transforms features; our nearest-centroid method performs classification.
9. **Why might classification accuracy stay low?** Too few originals, inconsistent framing/backgrounds, overlapping visual patterns, or a tiny test set.
10. **Why compare NumPy with scikit-learn?** To check that our mathematical implementation behaves like a standard implementation.

### Limitations to mention honestly
- Raw pixel PCA is sensitive to alignment, backgrounds, scale and pose.
- 16 × 16 images discard fine detail.
- Augmented copies are related to their source images and are not independent observations.
- A tiny test set makes accuracy unstable.
- PCA maximises overall variance, not class separation.
- Nearest-centroid is a simple baseline, not a state-of-the-art image classifier.

### Suggested live-coding plan (10–12 minutes)
1. Show one image and explain the 16 × 16 × 3 = 768 numbers.
2. Run centring and explain the mean image.
3. Live-code covariance and inspect its shape.
4. Live-code eigen-decomposition and sorting.
5. Project to 20 components and compare matrix shapes.
6. Reconstruct one image and change `k`.
7. Show the PCA map and compare with scikit-learn.

**Presentation punchline:** PCA does not “know Pokémon”. It discovers directions of variation in pixel data. We then inspect those directions and test whether they are useful for compression and a simple classification task.

## 17. PCA vs t-SNE: linear and nonlinear visualisation

Your assignment also asks for a comparison with **t-SNE**. We will use it only to make a 2D visualisation, not as a classifier and not as a replacement for PCA.

- **PCA** is linear: it finds straight-line directions that capture maximum variance.
- **t-SNE** is nonlinear: it tries to keep nearby points close together in a low-dimensional map.
- t-SNE axes do not have the same meaning as PC1 and PC2. Distances between far-apart clusters and the overall layout can be misleading.
- t-SNE is mainly a visualisation tool here. Do not compare its axes or explained variance with PCA.

For a fair, fast visual comparison, we first reduce the pixel features to at most 50 PCA features, then run t-SNE on those features.

In [ ]:
from sklearn.manifold import TSNE

# Combine training and held-out test images only for this visualisation.
# This does not fit the classifier or change the earlier held-out evaluation.
X_all_for_viz = np.vstack([X_train, X_test])
y_all_for_viz = np.concatenate([y_train, y_test])

# A compact PCA representation makes t-SNE faster and less noisy.
viz_k = min(50, X_all_for_viz.shape[0] - 1, X_all_for_viz.shape[1])
pca_viz = PCA(n_components=viz_k, svd_solver="full", random_state=RANDOM_STATE)
X_pca_viz = pca_viz.fit_transform(X_all_for_viz)

# t-SNE perplexity must be smaller than the number of samples.
perplexity = min(30, max(2, (len(X_all_for_viz) - 1) // 3))
if len(X_all_for_viz) >= 4:
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=RANDOM_STATE
    )
    X_tsne = tsne.fit_transform(X_pca_viz)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    classes = sorted(np.unique(y_all_for_viz))
    for label in classes:
        mask = y_all_for_viz == label
        axes[0].scatter(X_pca_viz[mask, 0], X_pca_viz[mask, 1],
                        label=label, alpha=0.75)
        axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                        label=label, alpha=0.75)
    axes[0].set_title("PCA: first two linear components")
    axes[1].set_title("t-SNE: nonlinear 2D map")
    for ax in axes:
        ax.set_xlabel("Dimension 1")
        ax.set_ylabel("Dimension 2")
        ax.grid(alpha=0.2)
    axes[1].legend()
    plt.suptitle("Same Pokémon images, two different visualisation methods")
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 4 images for this t-SNE demonstration.")

## 18. When can PCA struggle? A small nonlinear example

PCA looks for linear directions. It can struggle when the structure is curved or nested.

This tiny synthetic example uses two interleaving circles. The colours are labels for our eyes; PCA does not use them. t-SNE may reveal local neighbourhood structure more clearly, but remember: a visually separated t-SNE map is not proof of classification performance.

This section is deliberately separate from Pokémon classification. It explains *why a nonlinear method may sometimes be useful*.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler

X_rings, y_rings = make_circles(
    n_samples=500, factor=0.35, noise=0.06, random_state=RANDOM_STATE
)
X_rings = StandardScaler().fit_transform(X_rings)

rings_pca = PCA(n_components=2, random_state=RANDOM_STATE)
rings_pca_2d = rings_pca.fit_transform(X_rings)

rings_tsne = TSNE(
    n_components=2,
    perplexity=25,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE
).fit_transform(X_rings)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(rings_pca_2d[:, 0], rings_pca_2d[:, 1],
                c=y_rings, cmap="coolwarm", s=18)
axes[0].set_title("PCA on interleaving circles")
axes[1].scatter(rings_tsne[:, 0], rings_tsne[:, 1],
                c=y_rings, cmap="coolwarm", s=18)
axes[1].set_title("t-SNE on interleaving circles")
for ax in axes:
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.grid(alpha=0.2)
plt.suptitle("A nonlinear structure can be hard for linear PCA to unfold")
plt.tight_layout()
plt.show()

## 19. A cybersecurity application: network-flow feature reduction

PCA is not limited to pictures. A network-flow record can contain many numerical features, for example:

- duration of a connection
- bytes sent and received
- packets sent and received
- average packet size
- source and destination port numbers
- connection rate
- failed-connection count

A security analyst may have hundreds of numerical features. PCA can compress correlated features and help visualise unusual patterns.

### First: a self-contained synthetic demonstration

The data below is **synthetic**, not a real intrusion-detection dataset. We create normal-like and unusual network-flow patterns to demonstrate the workflow without requiring a download.

For real work, use an authorised dataset such as CIC-IDS2017 or a suitably anonymised flow export, and check its licence and feature documentation before use.

In [ ]:
# Create a toy network-flow dataset with correlated features.
# The labels are used only to colour the plot after PCA.
rng = np.random.default_rng(RANDOM_STATE)
n_normal, n_unusual = 350, 90

def make_flows(n, unusual=False):
    duration = rng.lognormal(mean=1.0 if not unusual else 1.4, sigma=0.45, size=n)
    packets_out = rng.poisson(lam=18 if not unusual else 55, size=n)
    packets_in = rng.poisson(lam=22 if not unusual else 7, size=n)
    bytes_out = packets_out * rng.normal(550 if not unusual else 1100, 90, size=n)
    bytes_in = packets_in * rng.normal(620 if not unusual else 180, 70, size=n)
    failed = rng.poisson(lam=0.3 if not unusual else 5, size=n)
    avg_packet = (bytes_out + bytes_in) / (packets_out + packets_in + 1)
    flow_rate = (packets_out + packets_in) / (duration + 0.1)

    return np.column_stack([
        duration, packets_out, packets_in, bytes_out,
        bytes_in, failed, avg_packet, flow_rate
    ])

flows_normal = make_flows(n_normal, unusual=False)
flows_unusual = make_flows(n_unusual, unusual=True)
X_flows = np.vstack([flows_normal, flows_unusual])
y_flows = np.array(["Normal"] * n_normal + ["Unusual-pattern"] * n_unusual)

feature_names = [
    "duration", "packets_out", "packets_in", "bytes_out",
    "bytes_in", "failed_connections", "average_packet_size", "flow_rate"
]
print("Network-flow matrix:", X_flows.shape)
print("Features:", feature_names)

### Standardise, apply PCA, and inspect the map

Unlike image pixels, network features have very different units and scales. Standardisation is important so that a feature measured in thousands does not dominate one measured in fractions merely because of its units.

We fit the scaler and PCA on the dataset used for this demonstration. For a proper predictive evaluation, split the real dataset first and fit both steps on training data only, just as we did in the Pokémon experiment.

In [ ]:
from sklearn.preprocessing import StandardScaler

flow_scaler = StandardScaler()
X_flows_scaled = flow_scaler.fit_transform(X_flows)

flow_pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z_flows = flow_pca.fit_transform(X_flows_scaled)

plt.figure(figsize=(8, 5))
for label in np.unique(y_flows):
    mask = y_flows == label
    plt.scatter(Z_flows[mask, 0], Z_flows[mask, 1],
                label=label, alpha=0.65, s=22)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA map of illustrative network-flow features")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print("Variance retained by two components:",
      f"{flow_pca.explained_variance_ratio_.sum():.1%}")

### Optional: apply the same workflow to your own cybersecurity CSV

If your team has a labelled network-flow CSV, set its path and label-column name below. The code selects numerical features, handles missing values, standardises them, and plots the first two PCs.

**Before using a real dataset:** remove identifiers and direct personal data; check for infinite values; understand which columns are labels versus features; and do not include the target label in PCA input. For a proper model evaluation, use a train/test split and fit preprocessing only on the training partition.

In [ ]:
CYBER_CSV_PATH = None       # Example: "network_flows.csv"
CYBER_LABEL_COLUMN = "label"

if CYBER_CSV_PATH is not None:
    import pandas as pd
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline

    cyber_df = pd.read_csv(CYBER_CSV_PATH)
    if CYBER_LABEL_COLUMN not in cyber_df.columns:
        raise ValueError(f"Label column '{CYBER_LABEL_COLUMN}' was not found.")

    y_cyber = cyber_df[CYBER_LABEL_COLUMN].astype(str).to_numpy()
    X_cyber_df = cyber_df.drop(columns=[CYBER_LABEL_COLUMN])
    X_cyber_df = X_cyber_df.select_dtypes(include=[np.number])

    if X_cyber_df.shape[1] < 2:
        raise ValueError("Need at least two numeric feature columns for a 2D PCA map.")

    X_cyber_values = X_cyber_df.replace([np.inf, -np.inf], np.nan).to_numpy()
    cyber_pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        PCA(n_components=2, random_state=RANDOM_STATE)
    )
    Z_cyber = cyber_pipeline.fit_transform(X_cyber_values)

    plt.figure(figsize=(8, 5))
    for label in np.unique(y_cyber):
        mask = y_cyber == label
        plt.scatter(Z_cyber[mask, 0], Z_cyber[mask, 1],
                    label=label, alpha=0.6, s=18)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA of numeric cybersecurity features")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("Set CYBER_CSV_PATH to run PCA on your own cybersecurity CSV.")

## 20. Final recap: what each method is doing

| Method | Main job in this project | What it does not do |
|---|---|---|
| NumPy PCA | Shows centring, covariance, eigenvectors, projection and reconstruction | Does not classify by itself |
| scikit-learn PCA | Checks our PCA implementation using a standard library | Does not guarantee good class separation |
| t-SNE | Creates a nonlinear 2D visualisation of local neighbourhoods | Is not our classifier; axes and global distances are not directly interpretable |
| Nearest centroid | Predicts a Pokémon species from PCA features | Is not part of PCA |
| Cybersecurity PCA | Compresses and visualises high-dimensional network features | Does not prove that a flow is malicious |

### Three takeaways
1. **PCA is linear dimensionality reduction:** it finds orthogonal directions that capture the greatest variance.
2. **PCA is useful for compression and exploration:** fewer components can retain much of the variation, but information is lost.
3. **PCA is not classification:** a separate classifier can use PCA features, and its performance must be evaluated on genuinely held-out data.

### Suggested presentation flow
- 2 min: show the Pokémon dataset and explain pixel vectors.
- 3 min: preprocessing, resizing, flattening and augmentation.
- 5 min: live-code centring, covariance, eigenvectors and projection.
- 3 min: scree plot, PCA map and reconstruction.
- 2 min: nearest-centroid classification and confusion matrix.
- 2 min: NumPy vs scikit-learn.
- 2 min: PCA vs t-SNE and nonlinear limitation.
- 1 min: cybersecurity application and limitations.